# 1、单工具的使用

## 1.1 使用ReAct模型


In [ ]:
from tavily import TavilyClient
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()

# 读取配置文件的信息
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")
# 获取Tavily搜索工具的实例
tavily_client = TavilyClient()

# --- 关键修改部分开始 ---

# 1. 定义一个包装函数，用于接收 Agent 传入的 query 字符串
def tavily_search_wrapper(query: str):
    """
    执行 Tavily 搜索并返回结果。
    """
    try:
        # search_depth 可以选 "basic" 或 "advanced"
        response = tavily_client.search(query=query, search_depth="basic")

        # Agent 通常只需要 'results' 部分的内容 (包含 url 和 content)
        # 直接返回整个 response 也可以，但 token 消耗会大一些
        return response.get('results', [])
    except Exception as e:
        return f"搜索出错: {str(e)}"

# 获取一个搜索的工具
# 方式1：
# search_tool = StructuredTool.from_function(
#     func=search.run,
#     name="Search",
#     description="用于检索互联网上的信息",
# )

# 2. 使用 Tool 包装这个函数
search_tool = Tool(
    name="Search",
    func=tavily_search_wrapper,  # 注意：这里只写函数名，不要加括号 ()
    description="用于检索互联网上的实时信息，例如天气、新闻等。",
)


# 获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 获取Agent的实例
agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION

# 获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools = [search_tool],
    llm = llm,
    agent = agent,
    verbose = True, # 显示详细的日志信息
)


# 通过AgentExecutor 调用invoke(),并得到响应
result = agent_executor.invoke("查询北京今天的天气情况")


# 处理响应数据
print(result)

In [ ]:
from tavily import TavilyClient
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
import os
import dotenv
from langchain_openai import ChatOpenAI
import json

# 加载环境变量
dotenv.load_dotenv()

# 读取配置文件的信息
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 获取Tavily实例
tavily_client = TavilyClient()

# --- 1. 定义实际执行搜索的函数 ---
def run_tavily_search(query: str) -> str:
    """
    执行互联网搜索并返回结果字符串。

    Args:
        query: 需要搜索的查询语句，例如 "北京天气" 或 "2024年奥运会金牌榜"
    """
    try:
        print(f"正在搜索: {query}") # 打印日志方便调试
        # search_depth="basic" 速度快，"advanced" 内容更深
        response = tavily_client.search(query=query, search_depth="basic")

        # 提取结果中的 results 列表
        results = response.get('results', [])

        # 将结果列表转换为字符串，方便大模型读取
        # 这里只取前2条以节省Token，实际使用可根据需求调整
        content_list = []
        for item in results[:3]:
            content_list.append(f"标题: {item['title']}\n链接: {item['url']}\n内容: {item['content']}")

        return "\n---\n".join(content_list)

    except Exception as e:
        return f"搜索发生错误: {str(e)}"

# --- 2. 使用 StructuredTool.from_function 创建工具 ---
search_tool = StructuredTool.from_function(
    func=run_tavily_search,
    name="Search",
    description="用于检索互联网上的实时信息，例如天气、新闻、股票等。输入应该是一个具体的查询字符串。",
    # infer_schema=True 默认是True，它会自动读取 run_tavily_search 的类型注解
)

# 获取大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 获取Agent的实例
# ZERO_SHOT_REACT_DESCRIPTION 适用于只能处理单个输入的工具
agent_type = AgentType.ZERO_SHOT_REACT_DESCRIPTION

# 获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=agent_type,
    verbose=True, # 显示思维链
    handle_parsing_errors=True # 容错处理
)

# 通过AgentExecutor 调用 invoke
try:
    # 这里的 key 建议使用 "input"
    result = agent_executor.invoke({"input": "查询今天星期几"})

    print("\n======== 最终回答 ========")
    print(result['output'])
except Exception as e:
    print(f"执行出错: {e}")


In [14]:
# pip install duckduckgo-search

from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain_openai import ChatOpenAI
from duckduckgo_search import DDGS # 确保引用的是这个
import os
import dotenv

# 加载环境变量 (主要用于 OpenAI，搜索不需要 Key 了)
dotenv.load_dotenv()
# 确保你有 OPENAI_API_KEY
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# --- 1. 修复后的 DuckDuckGo 搜索函数 ---
def run_ddg_search(query: str) -> str:
    """
    使用 DuckDuckGo 免费搜索引擎检索信息。
    Args:
        query: 搜索关键词
    """
    try:
        print(f"🔍 正在搜索: {query}")
        results = []

        # 实例化 DDGS 对象
        ddgs = DDGS()

        # 执行搜索
        # max_results: 限制返回结果数量
        # region="cn-zh": 优先中国地区结果
        ddg_gen = ddgs.text(query, region="cn-zh", timelimit="d", max_results=3)

        # 处理结果（注意：新版返回的是直接的列表或迭代器，不需要 context manager 也可以）
        if ddg_gen:
            for r in ddg_gen:
                results.append(f"标题: {r['title']}\n链接: {r['href']}\n摘要: {r['body']}")

        if not results:
            return "未找到相关结果。"

        return "\n---\n".join(results)

    except Exception as e:
        # 打印错误详情，方便调试
        return f"搜索出错: {str(e)}"

# --- 2. 创建 Tool ---
search_tool = StructuredTool.from_function(
    func=run_ddg_search,
    name="DuckDuckGo_Search",
    description="用于搜索互联网信息，不需要API Key。当不知道答案时使用此工具。",
)

# --- 3. 初始化 LLM ---
llm = ChatOpenAI(
    model="gpt-4o-mini", # 或者 gpt-3.5-turbo
    temperature=0,
)

# --- 4. 初始化 Agent ---
agent_executor = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True, # 能够看到思考过程
    handle_parsing_errors=True
)
# --- 5. 运行 ---
try:
    # 提问一个具时效性的问题来验证搜索能力
    input_query = "北京今天的天气怎么样？"

    print(f"用户问题: {input_query}")
    result = agent_executor.invoke({"input": input_query})

    print("\n======== 最终回答 ========")
    print(result['output'])

except Exception as e:
    print(f"运行出错: {e}")

用户问题: 北京今天的天气怎么样？


> Entering new AgentExecutor chain...
我需要查找北京今天的天气信息。  
Action: DuckDuckGo_Search  
Action Input: 北京今天的天气  🔍 正在搜索: 北京今天的天气


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\2075556447.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()



Observation: 标题: Toll Table – NLEX Corporation
链接: https://nlex.com.ph/toll-table/
摘要: This Privacy Policy describes how we – NLEX Corporation (NLEX) –collect, protect, use, disclose, and/or dispose of (Processing) the personal information of our customers, the motorists …
---
标题: Toll Fee Matrix Philippines
链接: https://toll.ph/matrix
摘要: Nov 1, 2025 · Philippine Toll Fee MatrixNorth Luzon Expressway • Subic-Clark-Tarlac Expressway
---
标题: 𝗡𝗟𝗘𝗫 𝗖𝗢𝗥𝗣. 𝗧𝗢 𝗖𝗢𝗟𝗟𝗘𝗖𝗧 𝗡𝗘𝗪 𝗔𝗗𝗝𝗨𝗦𝗧𝗘𝗗 𝗧𝗢𝗟𝗟 𝗙𝗘𝗘𝗦 𝗦𝗧𝗔𝗥𝗧𝗜𝗡𝗚 𝗢𝗡 𝗠𝗔𝗥𝗖𝗛 𝟮, 𝟮𝟬𝟮𝟱 ...
链接: https://trb.gov.ph/index.php/toll-rates/metro-manila-skyway-stage-3/47-press-release/83-2025-02-27-11-39-50
摘要: Feb 27, 2025 · New adjusted toll fees will be collected at the North Luzon Expressway (NLEX) starting on March 2, 2025. At the Open System (Balintawak, Caloocan, Mindanao Ave to North of …
Thought:我没有找到关于北京今天天气的信息。  
Action: DuckDuckGo_Search  
Action Input: 北京今天天气预报  🔍 正在搜索: 北京今天天气预报


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\2075556447.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()



Observation: 标题: Notepad++的替代品都有什么？ - 知乎
链接: https://www.zhihu.com/question/515834981
摘要: NDD 在MAC OS 12.3 操作系统 在我编写对比软件ccompare的过程中，我发现其核心功能，已经具备一个文本编辑器的雏形了。只需要再多花一部分额外的精力，便可以成为一个功能基本可以替换notepad++的简单文 …
---
标题: Notepad++ - 知乎
链接: https://www.zhihu.com/topic/19572058/intro
摘要: Notepad++是 Windows操作系统下的一套文本编辑器 (软件版权许可证: GPL)，有完整的中文化接口及支持多国语言编写的功能 (UTF8技术)。Notepad++功能比 Windows 中的 Notepad (记事本)强大，除了可以用来制 …
---
标题: Notepad++的开发者Don HO是一个怎样的人？ - 知乎
链接: https://www.zhihu.com/question/271281045?rf=271281041
摘要: 相信越来越多的人已经认识到notepad++作者的可恶了，我曾经也是notepad++的使用者，当我看到它的作者一个前台湾人，明目张胆的利用软件开始宣传他的错误观点后，便觉得要做些什么了。 notepad++不过是一 …
Thought:我仍然没有找到关于北京今天天气的信息。  
Action: DuckDuckGo_Search  
Action Input: 北京天气预报今天  🔍 正在搜索: 北京天气预报今天


...


Observation: 标题: フェイクポルノ - Wikipedia
链接: https://ja.wikipedia.org/wiki/フェイクポルノ
摘要: フェイクポルノ とは、 人工知能 (AI) にもとづく人物画像合成の技術である ディープフェイク を使った偽 ポルノ 動画のこと [1]。 2017年 11月2日 に、アメリカ合衆国の Reddit で「ディープフェ …
---
标题: Japanese Deepfake Hub
链接: https://japanese-deepfake-hub.jp/
摘要: Dec 5, 2025 · 日本人のアイドル (乃木坂46、欅坂46、日向坂46)や女優のディープフェイクをまとめたサイトです
---
标题: 【警鐘ルポ】「卒業アルバムの写真」から被害に…最新 ...
链接: https://news.yahoo.co.jp/articles/07e290af846c781cb5198623e88bbfabc4ca9237
摘要: 1 day ago · 今後は、各愛好家がそれぞれ精巧なフェイクポルノを″自給自足″する時代に入るでしょう」 フェイクポルノを提供するサーバーには小学生の ...
Thought:我仍然没有找到关于北京今天天气的信息。  
Action: DuckDuckGo_Search  
Action Input: 北京天气预报今天的情况  🔍 正在搜索: 北京天气预报今天的情况


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\2075556447.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()



Observation: 标题: Netflix, Prime Video, Hulu, HBO 等国外流媒体公司分别有 ...
链接: https://www.zhihu.com/question/309560443/answers/updated
摘要: Oct 22, 2021 · Amazon Prime Video还接收由Amazon Studios发行的电影并提供实况体育节目，此外，Prime Video还囊括了数百种获得许可的电影 …
---
标题: テレビ（ビエラ）でYouTube、Netflix、Prime Videoなどの ...
链接: https://jpn.faq.panasonic.com/app/answers/detail/a_id/43751/~/テレビ（ビエラ）でyoutube、netflix、prime-videoなどのアプリで動画が視聴できないときは
摘要: テレビ（ビエラ）でYouTube、Netflix、Prime Videoなどのアプリの視聴ができないときは、一時的にテレビの保護回路が働いている、インター …
---
标题: Netflix, Prime Video, Hulu, HBO 等国外流媒体公司分别有 ...
链接: https://www.zhihu.com/question/309560443
摘要: Oct 22, 2021 · 2. 本人对各个平台使用体验的喜好程度 Netflix>Amazon Prime Video>>>>>>>>>>>>>>>>>>>>>HBO GO Netflix …
Thought:我仍然没有找到关于北京今天天气的信息。  
Action: DuckDuckGo_Search  
Action Input: 北京今天天气情况  🔍 正在搜索: 北京今天天气情况


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\2075556447.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()



Observation: 标题: 今夜北京将有轻雾或雾 明夜至后天山区有小雪或零星小雪-资讯
链接: https://news.weather.com.cn/2025/12/4437580.shtml
摘要: 11 hours ago · 监测显示，昨天北京最高气温升至7℃左右。 今晨，北京体感寒冷。 （图/中国天气网王晓） 北京市气象台预计，未来三天云量逐渐增多，明天后半夜至后天上午天气转阴，山区有小雪或零星 …
---
标题: 北京将启动空气重污染黄色预警！明起气温下降，局地飘雪
链接: https://news.qq.com/rain/a/20251217A041JZ00
摘要: 11 hours ago · 北京市气象局11发布的天气预报， 今天（17日）下午晴间多云，北转南风二三级，最高气温7℃；夜间晴转多云，有轻雾或雾，南转北风一级左右 ...
---
标题: 北京今明两天夜间能见度较差，注意出行安全_京报网
链接: https://news.bjd.com.cn/2025/12/17/11468920.shtml
摘要: 11 hours ago · 今日天气 今天白天北京继续相约阳光，风力不大，最高气温在7℃上下，午间时段比较适宜大家户外活动和开窗通风，找个阳光充足的地方，晒晒太阳也是不错的。 夜幕降临，天空云量增 …
Thought:我现在找到了关于北京今天的天气信息。  
Final Answer: 北京今天白天晴间多云，最高气温约7℃，夜间有轻雾或雾，气温较低。

> Finished chain.

======== 最终回答 ========
北京今天白天晴间多云，最高气温约7℃，夜间有轻雾或雾，气温较低。


In [15]:

from langchain.agents import initialize_agent, AgentType
from langchain_community.tools import DuckDuckGoSearchRun # 直接导入官方工具
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 1. 直接实例化官方工具
search_tool = DuckDuckGoSearchRun()

# 2. LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3. Agent
agent_executor = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

# 4. 运行
try:
    result = agent_executor.invoke({"input": "北京今天天气"})
    print(result['output'])
except Exception as e:
    print(e)




> Entering new AgentExecutor chain...
I need to find the current weather in Beijing. 
Action: duckduckgo_search
Action Input: 北京今天天气

E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: Create an account or log in to Instagram - Share what you're into with the people who get you. Join Instagram! Sign up to see photos, videos, stories & messages from your friends, family & interests around the world. Instagramアカウントを作成、またはアカウントにログイン - 興味のあることを、あなたとつながっている人とシェアしよう。 Discover something new on Instagram and find what inspires you Reset your Instagram password by entering your email, phone number, or username.
Thought:It seems that the search did not return relevant information about the weather in Beijing. I need to refine my search query to get accurate weather information. 

Action: duckduckgo_search  
Action Input: 北京天气预报  

E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: 北京天气预报，及时准确发布中央气象台天气信息，便捷查询北京今日天气，北京周末天气，北京一周天气预报，北京蓝天预报，北京天气预报，北京40日天气预报，还提供北京的生活指数、健康指数、交通指数、旅 … 当前位置： 首页 北京市 北京天气预报 省份： 城市： 02:40更新 Nov 22, 2025 · 北京今日（9月16日）以多云天气为主，早晨至白天风力明显增大。 根据气象部门监测，今天白天北风三四级，阵风可达六七级，山区还可能出现短时阵雨。 全国天气网提供北京天气预报，未来北京10天天气，通过全国天气网详细了解北京天气预报以及北京周边各地区未来10天天气情况，温度，空气质量，降水，风力，气压，湿度，紫外线强度等！ 每小时本地天气预报、天气情况、降水、露点、湿度、大风 - 尽在 Weather.com 和 The Weather Channel
Thought:The search provided information about the weather in Beijing, indicating that today is mainly cloudy with significant wind. The north wind is expected to be at levels 3 to 4, with gusts reaching levels 6 to 7. There may also be brief rain showers in mountainous areas.

Final Answer: 北京今天天气以多云为主，北风三四级，阵风可达六七级，山区可能有短时阵雨。

> Finished chain.
北京今天天气以多云为主，北风三四级，阵风可达六七级，山区可能有短时阵雨。


上述示例也可以写为：


In [ ]:
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
dotenv.load_dotenv()

# 读取配置文件的信息
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

# 获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 获取Agent的实例
agent = AgentType.ZERO_SHOT_REACT_DESCRIPTION

# 获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools = [search],
    llm = llm,
    agent = agent,
    verbose = True, # 显示详细的日志信息
)


# 通过AgentExecutor 调用invoke(),并得到响应
result = agent_executor.invoke("查询北京今天的天气情况")


# 处理响应数据
print(result)


在 LangChain 中，创建 Agent 的方式经历了几个版本的演变。你提供的代码使用的是**最早期（Legacy）**的高级封装方式。

目前主要有 3 种 创建 Agent 的方法，我将按从“旧”到“新”的顺序为你介绍，并重点解释你代码中的参数含义。

### 方法 1：使用 initialize_agent (你代码中的方式)
这是 LangChain 最早期的“一键式”工厂函数。它把 Agent 的定义（Prompt构造）和 执行逻辑（AgentExecutor）封装在了一起。
⚠️ 状态： 已弃用 (Deprecated)，官方不再建议在新项目中使用，但仍需了解以维护旧代码。

参数详解

In [ ]:
agent_executor = initialize_agent(
    tools=[search_tool],            # 1. 工具箱：Agent 可以使用的武器
    llm=llm,                        # 2. 大脑：负责推理的 LLM
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, # 3. 代理类型（核心）
    verbose=True,                   # 4. 调试模式：打印思考过程
    handle_parsing_errors=True      # 5. 容错：防止格式错误导致崩溃
)

agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION:
- Zero-Shot: 不需要给模型提供示例（Few-shot），直接告诉它任务。
- ReAct: "Reasoning + Acting"（推理+行动）。模型会按照“思考 -> 选择工具 -> 观察结果 -> 再思考”的循环工作。
- Description: 模型完全依赖工具的 description 字段来决定是否使用该工具。
-
缺点：非常依赖 Prompt 的指令遵循能力，容易出现格式错误（所以必须开 handle_parsing_errors=True）。


### 方法 2：构造函数构建 (Standard / 0.1.0+ 版本)

在 LangChain v0.1 之后，官方推荐将 Agent（大脑/Prompt逻辑） 与 Executor（执行循环） 分开构建。这种方式比 initialize_agent 更透明，更容易定制 Prompt。
⚠️ 状态： 过渡方案（目前仍广泛使用，但在复杂场景下正逐渐被 LangGraph 取代）。

代码示例

In [ ]:
from langchain_openai import tools
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub # 从 LangChain Hub 拉取标准 Prompt

# 1. 拉取标准的 ReAct Prompt (也可以自己写)
prompt = hub.pull("hwchase17/react")

# 2. 创建 Agent (定义大脑如何工作)
agent = create_react_agent(llm, tools, prompt)

# 3. 创建 Executor (定义如何执行循环)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# 4. 运行
agent_executor.invoke({"input": "查询北京天气"})

- 优点：你可以修改 prompt，比如给它加上“你是一个海盗”的人设。
- 常用构建函数：
    - create_react_agent: 通用，适用于任何 LLM。
    - create_openai_tools_agent: 推荐。专用于 OpenAI 模型，利用原生的 Tool Calling 功能，比 ReAct 更稳定、更准确。

### 方法 3：使用 LangGraph (推荐 / 最新标准)

这是 LangChain 官方目前最推荐的方式。它不再把 Agent 看作一个黑盒循环，而是看作一个图（Graph）。

⚠️ 状态： 最新标准 (Production Ready)，解决了旧版 Agent 死循环、状态管理难、难以人为干预的问题。

代码示例

In [ ]:
from langgraph.prebuilt import create_react_agent

# LangGraph 把一切都简化了
# 这一行代码相当于上面的 initialize_agent + Executor
graph = create_react_agent(llm, tools)

# 运行 (输入格式变为 messages 列表)
inputs = {"messages": [("user", "查询北京天气")]}
result = graph.invoke(inputs)

## 1.2 使用FUNCTION_CALL模型

In [ ]:
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
dotenv.load_dotenv()

# 读取配置文件的信息
# os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")
os.environ['TAVILY_API_KEY'] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具的实例
search = TavilySearchResults(max_results=3)

# 获取一个搜索的工具
# 方式1：
# search_tool = StructuredTool.from_function(
#     func=search.run,
#     name="Search",
#     description="用于检索互联网上的信息",
# )

# 方式2：使用Tool
search_tool = Tool(
    func=search.run,
    name="Search",
    description="用于检索互联网上的信息",
)


# 获取大语言模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 获取Agent的实例 (使用function_call模型)
agent = AgentType.OPENAI_FUNCTIONS

# 获取AgentExecutor的实例
agent_executor = initialize_agent(
    tools = [search_tool],
    llm = llm,
    agent = agent,
    verbose = True, # 显示详细的日志信息
)


# 通过AgentExecutor 调用invoke(),并得到响应
result = agent_executor.invoke("查询北京今天的天气情况")


# 处理响应数据
print(result)

# 2、多工具的使用

需求：
- 计算特斯拉当前股价是多少？
- 比去年上涨了百分之几？（提示：调用PythonREPL实例的run方法）

## 2.1 使用ReAct模式

In [ ]:
# 1.导入相关依赖
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.python import PythonREPL


# 2. 设置 TAVILY_API 密钥
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")  # 需要替换为你的 Tavily API 密钥

# 3.定义搜索工具
search = TavilySearchResults(max_results=3)

search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息，特别是股票价格和新闻"
)

# 4.定义计算工具
python_repl = PythonREPL() # LangChain封装的工具类可以进行数学计算

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化"
)

# 5. 定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 6. 创建AgentExecutor执行器对象
agent_executor = initialize_agent(
    tools=[search_tool, calc_tool],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)


# 7. 测试股票价格查询
query = "比亚迪当前股价是多少？比去年上涨了百分之几？"
result=agent_executor.invoke(query)
print(f"查询结果: {result}")

## 2.2 使用FUNCTION_CALL模式

### 第一部分：核心概念解析

#### 1. 什么是 Function Calling (函数调用) 模型？

Function Calling 是 OpenAI GPT-3.5 和 GPT-4 系列模型的一项原生能力。

- 以前 (Text Completion): 你问模型“北京天气怎么样”，模型只能吐出一段中文文本“北京今天天气很好...”。如果你想用代码查天气，你得费劲地用正则表达式去抠这段字。
- 现在 (Function Calling): 你在请求模型时，顺便给它传一份工具说明书（JSON Schema）。如果模型觉得需要查天气，它不会乱说话，而是会返回一个标准的 JSON 对象，比如 {"name": "get_weather", "arguments": "{\"city\": \"Beijing\"}"}。
本质：模型被微调过，学会了将自然语言转换为 API 调用参数。它不执行代码，它只负责生成精准的参数。

#### 2. Function Calling vs. ReAct (旧模式) 的区别

特性	        ReAct (Reasoning + Acting)	                                         |   Function Calling (OpenAI Tools)
工作原理	    Prompt工程。靠写一段很长的提示词：“请按照 Thought: ... Action: ... 的格式回答”。|	模型微调。模型底层就“知道”可以调工具，通过专门的 API 参数 (tools) 传递。
稳定性	    差。模型经常忘了格式，或者少写个括号，导致程序正则解析失败。	                       |  极高。输出的是结构化 JSON，几乎不出错。
参数提取            	很难提取复杂的嵌套参数。	                                               |  可以轻松提取复杂的 JSON 对象（如列表、字典）。
幻觉问题	        容易编造不存在的工具名称。	                                                   | 严格遵循提供的 Schema，很少胡编乱造。


### 第二部分：实战场景设计
场景：用户想做一个“针对老年人的极简微信”。
Agent 任务流：
1. 用户提出想法。
2. Agent 觉得信息不够，决定调用 DuckDuckGo 搜索目前市场上的“适老化APP”有哪些痛点。
3. Agent 总结分析，生成一份 Markdown 格式的 PRD (产品需求文档)。
4. Agent 主动调用 FileSystem 工具，将文档保存到本地磁盘。

### 第三部分：方案一 —— 使用 LangChain 实现

LangChain 的 create_openai_tools_agent 完美封装了 Function Calling。



In [17]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import StructuredTool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pydantic import BaseModel, Field

# 加载配置
dotenv.load_dotenv()
# 确保环境变量中有 OPENAI_API_KEY
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# ==========================================
# 1. 定义工具 (多工具：一个搜索，一个保存)
# ==========================================

# --- 工具 A: 搜索工具 (直接用现成的) ---
search_tool = DuckDuckGoSearchRun(
    name="Market_Competitor_Search",
    description="专门用于搜索市场上的竞品、用户痛点或行业标准。在进行需求分析前，必须先使用此工具调研。"
)

# --- 工具 B: 本地文件保存工具 (自定义复杂工具) ---
# 定义参数结构 (使用 Pydantic 进行类型检查，这对于 Function Calling 很重要)
class SaveDocumentInput(BaseModel):
    file_name: str = Field(description="要保存的文件名称，必须以 .md 结尾，例如 'prd_v1.md'")
    content: str = Field(description="文件的完整文本内容，通常是 Markdown 格式")

def save_to_local_disk(file_name: str, content: str) -> str:
    """将文本内容写入本地文件系统"""
    try:
        if not file_name.endswith(".md"):
            file_name += ".md"

        # 模拟写入过程
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(content)
        return f"✅ 成功：文件已保存至 {os.path.abspath(file_name)}"
    except Exception as e:
        return f"❌ 失败：保存文件出错 {e}"

save_tool = StructuredTool.from_function(
    func=save_to_local_disk,
    name="Save_PRD_File",
    description="仅在需求分析完成，生成了完整文档后调用。用于将结果保存到本地。",
    args_schema=SaveDocumentInput # 绑定参数结构
)

# 工具列表
tools = [search_tool, save_tool]

# ==========================================
# 2. 构建 Prompt (赋予人设)
# ==========================================
system_prompt = """
你是一位拥有15年经验的资深产品经理。你的专长是将用户模糊的想法转化为专业的需求文档。

你的工作流程如下：
1. **理解需求**：仔细阅读用户的想法。
2. **市场调研**：必须调用 'Market_Competitor_Search' 工具，去看看市场上有没有类似产品，或者有什么痛点。
3. **撰写分析**：根据调研结果，撰写一份 PRD，包含：项目背景、用户画像、核心功能列表 (MVP)、技术风险。
4. **交付结果**：调用 'Save_PRD_File' 工具，将你写好的分析报告保存到本地。

注意：
- 必须基于搜索到的真实信息进行分析。
- 最终必须产生一个文件。
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), # 必须保留这个占位符，用于存放 Function Call 的中间结果
])

# ==========================================
# 3. 初始化 Agent
# ==========================================
# 必须使用支持 Tool Calling 的模型 (gpt-3.5-turbo-0125 或 gpt-4o)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 这一步会自动将 tools 转换为 OpenAI 的 JSON Schema 并绑定到 LLM
agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # 打开 verbose 可以清楚看到它什么时候调用了哪个工具
    handle_parsing_errors=True
)

# ==========================================
# 4. 运行
# ==========================================
if __name__ == "__main__":
    user_request = "我想做一个针对老年人的极简版微信，只有打电话和发照片功能。"

    print(f"🚀 开始处理需求: {user_request}\n")
    try:
        # 这里的 output 是 Agent 最后对用户说的话，而文件保存是过程中的动作
        result = agent_executor.invoke({"input": user_request})
        print("\n🤖 AI 最终回复:", result['output'])
    except Exception as e:
        print(f"出错: {e}")

🚀 开始处理需求: 我想做一个针对老年人的极简版微信，只有打电话和发照片功能。



> Entering new AgentExecutor chain...

Invoking: `Market_Competitor_Search` with `{'query': '极简版微信 老年人 通信应用'}`




E:\dingchuan\ai-demo\.venv\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


May 3, 2025 · 为了让中老年用户能更轻松、便捷地使用微信，微信团队花了不少心思，推出了一系列适老化功能，每一个都充满了巧思与温暖。 Aug 5, 2025 · 本篇文章从老年用户最核心的使用困境出发，结合极简交互设计理念，剖析适老化产品的改造策略，探讨如何通过“少即是多”的思维，真正打破老人与智能设备间的隔阂。 Apr 30, 2025 · 家里奶奶，一直用的是老年机，手机坏了，想给他用自己换下的手机，小米的，怕他不会用。 手机已经调成极简桌面了，有一键打电话的功能了。 Jun 18, 2025 · 微信、今日头条、美团等一大批老年人常用的APP已完成适老化改造，上线长辈版、关怀模式等符合老年人使用需求的专用版本，推出界面优化、语音交互、文字朗读、智能引导等创新功 … Jul 12, 2025 · 随着智能手机的普及，越来越多的老年人开始使用手机进行社交、支付、娱乐等日常活动。 然而，复杂的操作界面、过小的字体、繁琐的功能设置等问题，常常让老年人感到困扰。
Invoking: `Save_PRD_File` with `{'file_name': 'prd_simplified_wechat_for_seniors.md', 'content': '# 项目背景\n随着智能手机的普及，越来越多的老年人开始使用手机进行社交、支付和娱乐等日常活动。然而，复杂的操作界面和繁琐的功能设置常常让他们感到困扰。因此，开发一款针对老年人的极简版微信，专注于打电话和发送照片功能，将有助于提升他们的使用体验，满足他们的基本通信需求。\n\n# 用户画像\n- **年龄**：60岁及以上  \n- **技术水平**：对智能手机的使用较为陌生，习惯于传统的通信方式  \n- **需求**：希望能够方便地与家人和朋友保持联系，主要通过电话和照片分享  \n- **痛点**：现有的社交应用功能复杂，操作不便，容易造成使用障碍\n\n# 核心功能列表 (MVP)\n1. **一键拨号**：用户可以通过简单的界面快速拨打常用联系人电话。  \n2. **照片发送**：允许用户选择照片并发送给联系人，操作简单直观。  \n3. **语音提示**：在操作过程中提供语音提示，帮助用户理解每一步的操作。  \n4. **大字体和高对比度界面**：界面设计采用大字体和高对比度色彩，方便老年人阅

### 第四部分：方案二 —— 使用 OpenAI 原生 SDK 实现

这部分代码展示了脱离 LangChain，如何手动处理 tool_calls 循环。这能让你看到 Function Calling 到底传了什么数据。


完整代码

In [18]:
import os
import json
import dotenv
from openai import OpenAI
from duckduckgo_search import DDGS

dotenv.load_dotenv()
client = OpenAI()

# ==========================================
# 1. 定义 Python 函数 (实际干活的代码)
# ==========================================

def search_market(query):
    """搜索市场信息"""
    print(f"\n🔍 [Tool: Search] 正在搜索: {query}")
    try:
        ddgs = DDGS()
        results = ddgs.text(query, region="cn-zh", max_results=3)
        return json.dumps(results, ensure_ascii=False) if results else "未找到结果"
    except Exception as e:
        return f"搜索出错: {e}"

def save_file(filename, content):
    """保存文件"""
    print(f"\n💾 [Tool: Save] 正在保存文件: {filename}")
    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(content)
        return f"文件 {filename} 保存成功。"
    except Exception as e:
        return f"文件保存失败: {e}"

# 函数映射表 (用于后面根据名字调用函数)
available_functions = {
    "search_market": search_market,
    "save_file": save_file,
}

# ==========================================
# 2. 定义 Tool Schema (传给 GPT 的说明书)
# ==========================================
# 这就是 Function Call 的核心，必须严格按照 JSON Schema 格式写
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_market",
            "description": "搜索竞品、市场行情或用户痛点。",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "搜索关键词"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_file",
            "description": "将生成的内容保存到本地文件。",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "文件名，如 prd.md"},
                    "content": {"type": "string", "description": "文件内容"}
                },
                "required": ["filename", "content"]
            }
        }
    }
]

# ==========================================
# 3. 核心执行循环 (Agent Loop)
# ==========================================
def run_analyst_agent(user_prompt):
    # 初始化消息历史
    messages = [
        {"role": "system", "content": "你是一个严谨的产品经理。遇到需求先搜索(search_market)，分析后将结果保存(save_file)到本地。"},
        {"role": "user", "content": user_prompt}
    ]

    print(f"👤 用户: {user_prompt}")

    while True:
        # 1. 请求 GPT
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto" # 让模型自己决定用不用工具
        )

        response_msg = response.choices[0].message
        tool_calls = response_msg.tool_calls

        # 2. 判断模型是否想调工具
        if tool_calls:
            # 必须把模型的回复加入历史（包含 tool_calls 信息）
            messages.append(response_msg)

            # 可能一次调用多个工具，需要遍历
            for tool_call in tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                # 找到对应的 Python 函数并执行
                function_to_call = available_functions.get(fn_name)
                if function_to_call:
                    # 执行函数
                    fn_result = function_to_call(**fn_args)

                    # 将结果封装为 Tool Message 加入历史
                    messages.append({
                        "tool_call_id": tool_call.id, # 关键：必须对应 ID
                        "role": "tool",
                        "name": fn_name,
                        "content": str(fn_result)
                    })

            print("🤖 模型正在思考下一步...")
            # 循环继续，把工具结果发回给模型，看它是否还要继续调工具或输出回答
        else:
            # 模型不想调工具了，直接输出了回答
            print(f"\n✅ 最终回复: {response_msg.content}")
            break

if __name__ == "__main__":
    run_analyst_agent("我想做一个针对大学生的二手书交易平台小程序。")

👤 用户: 我想做一个针对大学生的二手书交易平台小程序。

🔍 [Tool: Search] 正在搜索: 大学生 二手书 交易平台 小程序


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\3327788906.py:18: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()


🤖 模型正在思考下一步...

🔍 [Tool: Search] 正在搜索: 二手书交易平台 市场分析


C:\Users\rates\AppData\Local\Temp\ipykernel_23856\3327788906.py:18: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS()


🤖 模型正在思考下一步...

💾 [Tool: Save] 正在保存文件: 大学生二手书交易平台调研.md

💾 [Tool: Save] 正在保存文件: 二手书交易平台市场分析.md
🤖 模型正在思考下一步...

✅ 最终回复: 我已经进行了市场调研，并将结果保存为两个文件：

1. **大学生二手书交易平台调研.md**
   - 包含竞品分析、市场现状和用户痛点分析。

2. **二手书交易平台市场分析.md**
   - 包含行业前景、市场供需和品牌竞争情况的详细分析。

如需进一步的信息或其他方面的支持，请告知我！


### 总结
1. 复杂性体现：你可以看到 Agent 不是“问一句答一句”，而是经历了 用户 -> 搜索 -> 思考 -> 撰写 -> 保存 -> 回复 的长链路。
2. LangChain 版：帮你省去了 while 循环、JSON Schema 定义、参数解析等繁琐工作，更适合生产环境快速开发。
3. OpenAI 原生版：让你彻底明白了底层逻辑——所谓 Agent，本质上就是一个不断把函数执行结果“喂”回给 LLM 的 While True 循环。

In [2]:
# 1.导入相关依赖
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.python import PythonREPL


# 3.定义搜索工具
search = TavilySearchResults(tavily_api_key=os.getenv("TAVILY_API_KEY"),max_results=3)

search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息，特别是股票价格和新闻"
)

# 4.定义计算工具
python_repl = PythonREPL() # LangChain封装的工具类可以进行数学计算

calc_tool = Tool(
    name="Calculator",
    func=python_repl.run,
    description="用于执行数学计算，例如计算百分比变化"
)

# 5. 定义LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 6. 创建AgentExecutor执行器对象
agent_executor = initialize_agent(
    tools=[search_tool, calc_tool],
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS, #唯一需要修改的位置
    verbose=True
)


# 7. 测试股票价格查询
query = "比亚迪当前股价是多少？比去年上涨了百分之几？"
result=agent_executor.invoke(query)
print(f"查询结果: {result}")

AssertionError: PythonREPL has been deprecated from langchain_community due to being flagged by security scanners. See: https://github.com/langchain-ai/langchain/issues/14345 If you need to use it, please use the version from langchain_experimental. from langchain_experimental.utilities.python import PythonREPL.

# 3、自定义函数与工具

举例：计算3的平方，Agent自动调用工具完成

In [19]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain_openai import ChatOpenAI
import langchain


# 1. 定义工具 - 计算器（要求字符串输入）
def simple_calculator(expression: str) -> str:
    """
    基础数学计算工具，支持加减乘除和幂运算
    参数:
        expression: 数学表达式字符串，如 "3+5" 或 "2**3"
    返回:
        计算结果字符串或错误信息
    """
    print(f"\n[工具调用] 计算表达式: {expression}")

    print("只因为在人群中多看了你一眼，确认下你调用了我^_^")
    return str(eval(expression))


# 2. 创建工具对象
math_calculator_tool = Tool(
    name="Math_Calculator",  # 工具名称（Agent将根据名称选择工具）
    func=simple_calculator,  # 工具调用的函数
    description="用于数学计算，输入必须是纯数学表达式（如'3+5'或'3**2'表示平方）。不支持字母或特殊符号"  # 关键：明确输入格式要求
)

# 3. 初始化大模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# 4. 初始化AgentExecutor（使用零样本React模式、增加超时设置）
agent_executor = initialize_agent(
    tools=[math_calculator_tool],  # 可用的工具列表
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # 简单指令模式
    verbose=True  # 关键参数！在控制台显示详细的推理过程
)

# 5. 测试工具调用（添加异常捕获）
print("=== 测试：正常工具调用 ===")
response = agent_executor.invoke("计算3的平方")  # 向Agent提问
print("最终答案:", response)

=== 测试：正常工具调用 ===


> Entering new AgentExecutor chain...
我需要计算3的平方，这可以用数学表达式3**2来表示。  
Action: Math_Calculator  
Action Input: 3**2  
[工具调用] 计算表达式: 3**2
只因为在人群中多看了你一眼，确认下你调用了我^_^

Observation: 9
Thought:我现在知道最终答案  
Final Answer: 9

> Finished chain.
最终答案: {'input': '计算3的平方', 'output': '9'}
